#### 1. 데이터 구조 이해

- **하나의 JSON 파일**  
  - 모든 이미지 메타데이터, 객체 Annotation, 카테고리, 라이선스 정보가 하나의 JSON 파일(예: `instances_train2017.json` 또는 `instances_val2017.json`)에 저장됨

- **주요 키 구성**  
  - **images:** 각 이미지의 파일명, 고유 ID, 해상도 등의 메타데이터  
  - **annotations:** 각 이미지에 포함된 객체의 Annotation 정보(바운딩 박스, segmentation, keypoint 등)  
  - **categories:** 사용되는 객체 카테고리의 이름, ID, 상위 카테고리 정보  
  - **licenses:** 이미지 사용에 관련된 라이선스 정보

---

#### 2. Annotation의 유형

- **바운딩 박스 (Bounding Box):**  
  - 객체의 위치와 크기를 나타내는 사각형 정보

- **Instance Segmentation:**  
  - 객체의 경계를 보다 세밀하게 표현하기 위해 polygon(다각형) 또는 마스크 형태의 정보 제공

- **Keypoint Detection:**  
  - 사람의 특정 관절이나 부위의 좌표 정보를 포함 (주로 사람 포즈 추정을 위해 사용)

---

#### 3. 데이터셋 분할 및 버전

- **분할 (Train/Val/Test):**  
  - 학습, 검증, 평가 용도로 분리되어 있으며, 실습 시 데이터 분할에 따라 적절한 셋을 사용

- **버전 차이:**  
  - 2014, 2017 등 여러 버전이 있으며, 각 버전마다 이미지 및 Annotation 수가 다를 수 있음

---

#### 4. COCO API 활용

- **pycocotools 라이브러리:**  
  - COCO 데이터를 쉽게 로드하고 탐색, 시각화할 수 있도록 도와주는 API 제공  
  - 설치 시 pip보다 conda 또는 GitHub 소스코드를 이용하는 방법 추천

- **주요 기능**  
  - `COCO()` 생성자를 통해 JSON 파일을 로드  
  - `getCatIds()`, `loadCats()`: 카테고리 관련 정보 확인  
  - `getImgIds()`, `loadImgs()`: 이미지 메타데이터 조회  
  - `getAnnIds()`, `loadAnns()`: 특정 이미지나 카테고리에 대한 Annotation 정보 조회  
  - `showAnns()`: Annotation 시각화 지원

---

#### 5. 실습 시 유의사항

- **데이터 불균형:**  
  - 일부 카테고리는 데이터 수가 적을 수 있으므로, 모델 학습 시 주의가 필요함

- **Annotation 품질:**  
  - Annotation은 수작업으로 생성된 경우도 있으므로, 일부 부정확할 수 있음

- **라이선스 준수:**  
  - 데이터를 사용할 때 라이선스 정보를 확인하고 적절히 인용





In [ ]:
#@title 데이터 다운로드
from pycocotools.coco import COCO
import numpy as np

In [ ]:
!mkdir -p /content/data

In [ ]:
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip annotations_trainval2017.zip -d /content/data

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip
!unzip val2017.zip -d /content/data

In [ ]:
datadir = '/content/data'
datatype = 'val2017'
annfile = f'{datadir}/annotations/instances_{datatype}.json'
print(annfile)

JSON 파일 확인하기 (명령어 버전)
아래는 각 명령어가 수행하는 작업에 대한 설명이다.

1. **`!ls -lia /content/data/annotations/instances_val2017.json`**  
   - `ls` 명령어로 해당 파일의 상세 정보를 보여준다.  
   - 옵션:  
     - `-l`: 긴(long) 형식으로 파일 정보를 표시 (파일 권한, 소유자, 크기 등).  
     - `-i`: 파일의 inode 번호를 함께 표시.  
     - `-a`: 숨김 파일을 포함하여 모든 파일을 보여줌 (여기서는 파일 하나만 지정했으므로 크게 달라지지 않음).

2. **`!sudo apt-get install jq`**  
   - 시스템에 `jq`라는 JSON 처리 도구를 설치한다.  
   - `sudo`: 관리자 권한으로 실행합니다.  
   - `apt-get install jq`: 패키지 관리자를 통해 `jq` 패키지를 설치한다.

3. **`!jq . /content/data/annotations/instances_val2017.json > output.json`**  
   - `jq`를 사용하여 JSON 파일(`instances_val2017.json`)의 내용을 읽고,  
   - `.` (전체 내용을) 파싱한 후,  
   - 결과를 `output.json` 파일로 저장한다.  
   - 이를 통해 JSON 파일을 사람이 읽기 쉬운 포맷으로 변환할 수 있다.

4. **`!head -200 output.json`**  
   - `head` 명령어로 `output.json` 파일의 처음 200줄을 출력한다.  
   - 파일의 상단 부분을 확인할 때 유용함.

5. **`!tail -600 output.json`**  
   - `tail` 명령어로 `output.json` 파일의 마지막 600줄을 출력한다.  
   - 파일의 하단 부분을 확인할 때 사용한다.

6. **`!grep -n 'annotations' output.json`**  
   - `grep` 명령어로 `output.json` 파일 내에서 `'annotations'`라는 문자열을 검색한다.  
   - 옵션 `-n`은 해당 문자열이 나타난 줄의 번호와 함께 결과를 출력한다.

7. **`!head -50300 output.json | tail -300`**  
   - 먼저 `head -50300 output.json`을 통해 `output.json` 파일의 처음 50,300줄을 가져온다.  
   - 그 후, 파이프(`|`)를 사용하여 그 출력 결과에서 `tail -300` 명령어로 마지막 300줄만 출력한다.  
   - 결과적으로, 파일의 50,001번째 줄부터 50,300번째 줄까지의 내용을 확인할 수 있게 된다.

이와 같이 각 명령어는 파일의 상세 정보 확인, JSON 파일의 포맷팅, 그리고 특정 부분의 내용을 검토하는 데 사용됨.

In [ ]:
!ls -lia /content/data/annotations/instances_val2017.json

In [ ]:
!ls -al /content/data/annotations/

In [ ]:
!sudo apt-get install jq

In [ ]:
!jq . /content/data/annotations/instances_val2017.json > output.json

In [ ]:
!head -200 output.json

In [ ]:
!tail -600 output.json

In [ ]:
!grep -n 'annotations' output.json

In [ ]:
!head -50300 output.json | tail -300

In [ ]:
#@title Json 파일 확인하기

import os
import json

# 파일 정보 확인
file_path = '/content/data/annotations/instances_val2017.json'
try:
    stat_info = os.stat(file_path)
    print("파일 크기(바이트) ", stat_info.st_size)
    print("수정 시간 : ", stat_info.st_mtime)
    print("생성 시간 : ", stat_info.st_ctime)
    print("Inode 번호", stat_info.st_ino)

except FileNotFoundError:
    print("파일이 존재하지 않습니다. ", file_path)

#json 파일을 사람이 읽기 편한 형태로 저장

with open(file_path, 'r', encoding = 'utf-8') as f:
    data = json.load(f)
print(type(data))

with open('output2.json', 'w', encoding = 'utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)




In [ ]:
#output2.json 파일의 처음 200개
with open('output2.json', 'r', encoding='utf-8') as f:
    for _ in range(200):
        line = f.readline()
        if not line:
            break
        print(line, end='')

In [ ]:
with open('output2.json', 'r', encoding='utf-8') as f:
    lines = f.readlines()
for line in lines[-200:]:
    print(line, end='')

In [ ]:
#@title coco API
from pycocotools.coco import COCO
coco = COCO(annfile)

In [ ]:
#@title category info
print(coco.getCatIds())

In [ ]:
cats = coco.loadCats(coco.getCatIds())
cats

In [ ]:
#@title Category and Super category
nms = [cat['name'] for cat in cats]
print(nms)
supnms = [cat['supercategory'] for cat in cats]
print(supnms)

In [ ]:
#@title 지정된 이미지를 데이터셋에서 로드하기
catIds = coco.getCatIds(catNms=['person', 'dog', 'skateboard'])
imgIds = coco.getImgIds(catIds=catIds)
print(catIds)
imgIds = coco.getImgIds(catIds=catIds)

In [ ]:
#@title 이미지 확인
img = coco.loadImgs(imgIds[1])[0]
print(img)

coco_url = img['coco_url']
print(coco_url)

In [ ]:
import urllib.request

def download_image(url, target_path):
    urllib.request.urlretrieve(url, target_path)

download_image(img['coco_url'], '/content/data/' + img['file_name'])

In [ ]:
import cv2
import matplotlib.pyplot as plt
import pylab

file_path = '/content/data/' + img['file_name']

image_array = cv2.imread(file_path)
image_array = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(image_array)
pylab

#### Instance Segmentation 시각화 - COCO API 활용한 시각화
- getAnnIds()로 특정 image에 해당하는 annotation id를 가져온 후에 이 id를 loadAnns()로 입력하여 해당 이미지의 모든 annotation 정보를 가져옴.
- segmentation 정보는 polygon 형태로 되어 있음.
- annotation 정보를 coco.showAnns(anns)에 입력하여 instance segmentation 시각화 수행.

In [ ]:
annIds = coco.getAnnIds(imgIds=img['id'], catIds=catIds, iscrowd=None)
annIds


In [ ]:
anns = coco.loadAnns(annIds)
anns

1. 구조

    - 'segmentation' 키의 값은 리스트이며, 그 내부에 또 다른 리스트가 있음.
    - 내부 리스트는 객체의 경계선을 이루는 좌표값들의 나열이다.
2. 좌표값 나열

    - 내부 리스트의 값들은 순서대로 [x1, y1, x2, y2, x3, y3, ...] 형태로 구성되어 있다.
    - 예를 들어, 처음 두 값 228.43, 247.9는 첫 번째 점의 x, y 좌표를 의미합니다.
    - 이어지는 229.63, 206.62는 두 번째 점의 좌표.
3. 다각형 형성
    - 이 좌표들을 순서대로 연결하면 객체의 경계가 이루는 다각형이 만들어짐.
    - 마지막 점과 첫 번째 점을 연결하면 닫힌 경계가 형성되어 객체의 모양을 정의하게 된다.

In [ ]:
# showAnns( )는 annotation 정보들을 입력 받아서 Visualization 시켜줌. 단 먼저 matplotlib 객체로 원본 이미지가 먼저 로드되어 있어야 함.
plt.figure(figsize=(12, 14))
plt.imshow(image_array)
plt.axis('off')

# coco.showAnns(anns)는 불러온 annotation 정보(anns)를 기반으로 segmentation 경계 등을 이미지 위에 그려줌.
coco.showAnns(anns)


In [ ]:
len(anns), type(anns[0])

In [ ]:
print(anns[2]['segmentation'])
ann2seg = anns[2]['segmentation'][0]
print(type(ann2seg), len(ann2seg))
print(ann2seg)

In [ ]:
polygon_x = [x for index , x in enumerate(ann2seg) if index % 2 == 0]
polygon_y = [y for index , y in enumerate(ann2seg) if index % 2 == 1]

print("polygon_x : ", polygon_x)
print("polygon_y : ", polygon_y)

polygon_xy = [[x, y] for x, y in zip(polygon_x, polygon_y)]
print("polygon_xy : ", polygon_xy)



#### polylines- open cv
- OpenCV의 `cv2.polylines` 함수를 사용해 다각형 외곽선을 녹색으로 그려 이미지를 업데이트.  
- 최종적으로, matplotlib을 사용하여 외곽선이 표시된 이미지를 시각화.

- **다각형 외곽선 그리기**

   ```python
   draw_img = cv2.polylines(draw_img, [polygon_xy], True, (0, 255, 0))
   ```
   - **설명:**  
     - `cv2.polylines` 함수는 이미지에 다각형의 외곽선을 그리는 OpenCV 함수.
     - **인자 설명:**  
       - 첫 번째 인자 `draw_img`: 외곽선을 그릴 대상 이미지.
       - 두 번째 인자 `[polygon_xy]`: 다각형의 좌표 배열을 리스트 형태로 전달. (여러 개의 다각형을 동시에 그릴 수 있음)
       - 세 번째 인자 `True`: 다각형을 닫힌(closed) 형태로 그릴지 여부를 지정. `True`이면 마지막 점과 첫 번째 점이 자동으로 연결되어 닫힌 도형이 된다.
       - 네 번째 인자 `(0, 255, 0)`: 외곽선의 색상(BGR 포맷)입니다. 여기서는 녹색으로 지정되어 있다.
     - 이 함수 호출로 `draw_img`에 다각형 외곽선이 그려진 결과를 저장.

In [ ]:
import numpy as np

# opencv의 polylines를 이용하여 변환된 polygon 좌표 적용하여 instance segmentation 외곽선 시각화
green_color = (0, 255, 0)

draw_img = image_array.copy()
polygon_xy = np.array(polygon_xy, np.int32)
draw_img = cv2.polylines(draw_img, [polygon_xy], True, (0, 255, 0))

plt.figure(figsize=(12, 14))
plt.imshow(draw_img)
plt.axis('off')

#### OpenCV의 `fillPoly`
- **다각형 내부 영역 채우기**

   ```python
   draw_img = cv2.fillPoly(draw_img, [polygon_xy], (0, 255, 0))
   ```
   - **설명:**  
     - `cv2.fillPoly` 함수는 지정된 다각형 내부를 특정 색상으로 채운다.
     - **매개변수 설명:**  
       - 첫 번째 인자 `draw_img`: 채우기를 적용할 대상 이미지.
       - 두 번째 인자 `[polygon_xy]`: 다각형 좌표 배열을 리스트 형태로 전달.  
         - 여러 개의 다각형을 동시에 채울 수 있으므로 리스트로 감싸준다.
       - 세 번째 인자 `(0, 255, 0)`: 채우기에 사용할 색상(BGR 포맷). 여기서는 녹색을 사용.
     - 함수 실행 후, 다각형 내부 영역이 녹색으로 채워진 이미지가 반환되어 `draw_img`에 저장.


In [ ]:
# opencv의 fillPoly를 이용하여 변환된 polygon 좌표 적용하여 instance segmentation 내부 시각화
green_color = (0, 255, 0)

draw_img = image_array.copy()
polygon_xy = np.array(polygon_xy, np.int32)
draw_img = cv2.fillPoly(draw_img, [polygon_xy], (0, 255, 0))

plt.figure(figsize=(12, 14))
plt.imshow(draw_img)
plt.axis('off')

이 코드는 COCO API에서 제공하는 `annToMask()` 함수를 사용하여, 주어진 annotation의 polygon 정보를 이진 마스크(binary mask) 형태로 변환하는 과정을 보여준다.

- `annToMask()`를 사용하여 annotation의 polygon 정보를 이진 마스크로 변환.
-  **mask 생성**  
   ```python
   mask = coco.annToMask(anns[2])
   ```
   - **설명:**  
     - `coco.annToMask()` 함수는 주어진 annotation 정보를 기반으로 해당 객체의 영역을 나타내는 이진 마스크를 생성.
     - 여기서 `anns[2]`는 annotation 리스트의 세 번째 요소를 의미하며, 이 annotation의 polygon segmentation 정보를 사용하여 mask를 만든다.
     - 결과로 반환되는 `mask`는 원본 이미지와 동일한 높이와 너비를 가지는 2차원 배열로, 객체의 픽셀이 있는 영역은 1(또는 True), 나머지 영역은 0(또는 False)로 표시된다.

-  **마스크 내의 픽셀 값 통계 출력**  
   ```python
   print('0보다 큰 값이 있는 mask shape:', mask[mask > 0].shape, '0이 있는 mask shape:', mask[mask == 0].shape)
   ```
   - **설명:**  
     - `mask[mask > 0]`는 mask 배열에서 값이 0보다 큰 픽셀들을 선택. 이 영역은 객체가 존재하는 영역을 의미함.
     - `mask[mask == 0]`는 mask 배열에서 값이 0인 픽셀들을 선택하며, 객체가 없는 배경 영역을 의미함.
     - 두 출력값을 통해 객체 영역과 배경 영역의 픽셀 수를 파악할 수 있다.

In [ ]:
mask = coco.annToMask(anns[2])
print(mask.shape)
print(draw_img.shape)

print("0보다 큰 값이 있는 마스크 shape : ", mask[mask>0].shape)
print("0보다 큰 값이 있는 마스크 shape : ", mask[mask==0].shape)
print(mask[mask>0])

In [ ]:
plt.figure(figsize=(12, 14))
plt.imshow(mask)
plt.axis('off')

In [ ]:
# 빈 마스크 생성
# image_array.shape[0:2]는 원본 이미지의 높이와 너비를 의미.
# np.zeros(...)를 이용해 동일한 크기의 2차원 배열을 생성.
zero_mask = np.zeros(image_array.shape[0:2])

In [ ]:
# 다각형 영역을 마스크에 채우기
masked_polygon = cv2.fillPoly(zero_mask, [polygon_xy], 1)
masked_polygon

In [ ]:
## polygon 정보를 mask 정보로 변환.
zero_mask = np.zeros(image_array.shape[0:2]) # mask 정보는 2차원 선호
masked_polygon = cv2.fillPoly(zero_mask, [polygon_xy], 1)

print('zero_mask shape:', zero_mask.shape, 'mask shape:', masked_polygon.shape)
print('masked_polygon 0보다 큰 값:', masked_polygon[masked_polygon > 0])
print(masked_polygon[masked_polygon > 0].shape, masked_polygon[masked_polygon == 0].shape)

plt.figure(figsize=(12, 14))
plt.imshow(masked_polygon)
plt.axis('off')

In [ ]:
#@title 마스크 블렌딩

# mask 값에 따라 channel 별로 alpha값을 감안하여 색상 적용
def apply_mask_01(image, mask, color, alpha=0.5):
    for c in range(3):
        image[:, :, c] = np.where(mask == 1,
                              image[:, :, c] *
                              (1 - alpha) + alpha * color[c] * 255,
                              image[:, :, c])
    return image

In [ ]:
# 보통 bool 값으로 masking 정보 제공.
masked_bool = masked_polygon.astype(bool)
print('masked_bool shape:', masked_bool.shape)
print(masked_bool[masked_bool==1].shape, masked_bool[masked_bool==0].shape)
print(masked_bool[masked_bool==1])


- **각 채널에 대해 마스크 적용 및 색상 blending**  
   ```python
   for c in range(3):
       image[:, :, c] = np.where(mask == 1,
                                 image[:, :, c] * (1 - alpha) + alpha * color[c] * 255,
                                 image[:, :, c])
   ```
   - **반복문:**  
     - `for c in range(3):`를 통해 이미지의 각 컬러 채널(B, G, R)을 순차적으로 처리.
   - **np.where 사용:**  
     - `np.where(condition, value_if_true, value_if_false)` 함수를 사용하여 마스크 값에 따라 각 픽셀의 값을 선택.
     - 조건: `mask == 1`  
       - 마스크의 값이 1인 픽셀, 즉 객체 영역에 해당하면 두 번째 인자의 계산 결과를 적용.
       - 그렇지 않으면(배경 영역) 원래의 픽셀 값을 그대로 유지.
   - **색상 blending 계산:**  
     - `image[:, :, c] * (1 - alpha) + alpha * color[c] * 255`  
       - 원본 이미지 픽셀 값에 `(1 - alpha)`를 곱해 투명도를 적용하고,
       - 지정한 색상(color[c])에 `alpha`와 255(픽셀 최대값)를 곱해 blending에 기여하는 색상 값을 계산.
       - 두 값을 더해 최종적으로 해당 픽셀의 채널 값이 결정.



In [ ]:
draw_img = image_array.copy()
masked_image = apply_mask_01(draw_img, masked_bool, (0,255,0), alpha = 0.6)
plt.figure(figsize=(12, 14))
plt.imshow(masked_image)
plt.axis('off')

In [ ]:
def apply_mask_02(image, mask, color, alpha=0.5):
  """Apply the given mask to the image.
  """
  image = np.where(mask == 1, color, image)
  return image

In [ ]:
 np.stack([masked_bool, masked_bool, masked_bool], axis=2).shape


In [ ]:
draw_img = image_array.copy()
stacked_mask = np.stack([masked_bool, masked_bool, masked_bool], axis=2)
masked_image = apply_mask_02(draw_img, stacked_mask, (0, 255, 0))
plt.figure(figsize=(12, 14))
plt.imshow(masked_image)
plt.axis('off')